In [2]:
try:
    import xlearn
except:
    !pip install -q -U xlearn
    import xlearn

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for xlearn (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xlearn
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> xlearn


ModuleNotFoundError: No module named 'xlearn'

In [ ]:
!pip show gcc cmake
print()
!pip show xlearn

In [ ]:
## -- System dependencies --
import sys, os, gc
import torch

## -- Device-Agnostic (GPU/CPU) --
def get_system_info():
    if torch.cuda.is_available():
        return f"GPU: {torch.cuda.get_device_name(0)}"
    else:
        return f"CPU: {os.cpu_count()}"

get_system_info()

In [ ]:
## -- Data Manipulation --
import numpy as np, pandas as pd, random

## -- Visualization --
from IPython.display import display, Image
import matplotlib.pyplot as plt
import seaborn as sns

## -- Functional Tools --
import shutil, itertools
from tqdm.auto import tqdm
from time import time, sleep

## -- Machine Learning --
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import CalibrationDisplay, CalibratedClassifierCV
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize, TargetEncoder as skTE
from sklearn.metrics import roc_auc_score, log_loss, ConfusionMatrixDisplay, RocCurveDisplay

import warnings
warnings.simplefilter('ignore')
warnings.filterwarnings('ignore')

## -- Set Configuration --
# sklearn.set_config(transform_output="pandas")
# pd.options.mode.copy_on_write = True
pd.set_option("display.max_columns", 100)
# plt.style.use('ggplot')

warnings

In [ ]:
TEAL, GOLD, CORAL, BLUE, DARK, CRIM, MUTED, INK, PAPER = '#168c91','#e8b44f','#dc705a',\
                                                         'dodgerblue','#0B4F7A','crimson',\
                                                         '#627D98','#102A43','#faf9f5'
plt.rcParams.update({
    'figure.facecolor': PAPER,
    'axes.facecolor': PAPER,
    'savefig.facecolor': PAPER,
    'font.family': 'DejaVu Sans',
    'font.size': 9,
    'axes.titlesize': 11,
    'axes.titleweight': 'semibold',
    'axes.labelcolor': INK,
    'text.color': INK,
    'xtick.color': DARK,
    'ytick.color': DARK,
    'axes.edgecolor': MUTED,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.color': MUTED,
    'patch.edgecolor': DARK,
    # 'patch.force_edgecolor': True,
    # 'patch.linewidth': 1.5,
    # 'figure.dpi':110,
    # 'savefig.dpi':160,
})

In [ ]:
## -- ⚠️ IMPORTANT: SELECT PLATFORM ⚠️ --
PLATFORM = "kaggle" # -> 'colab' 'kaggle'

if PLATFORM == "kaggle":
    PATH   = "/kaggle/input/competitions/playground-series-s6e11/"
    submit = pd.read_csv(PATH+"sample_submission.csv")
    train  = pd.read_csv(PATH+"train.csv")
    test   = pd.read_csv(PATH+"test.csv")
    ORIG_PATH = "/kaggle/input/datasets/uditjain13/credit-card-fraud-detection-2026/"
    orig = pd.read_csv(ORIG_PATH+"credit_card_fraud_2026.csv")
elif PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PATH   = "/content/drive/MyDrive/-- shared_notebooks --/Ps6e11 | Credit Card Fraud/_fraud_data/"
    submit = pd.read_csv(PATH+"sample_submission.csv")
    train  = pd.read_csv(PATH+"train.csv")
    test   = pd.read_csv(PATH+"test.csv")
    orig   = pd.read_csv(PATH+"credit_card_fraud_2026.csv")

TARGET, ID = 'is_fraud', 'id'

## -- Check if id is unique --
if ID in train.columns:
    if train.id.is_unique:
        print(f"Train ID column is unique: {train.id.is_unique}... Dropped ✅!")
        train = train.drop(columns=[ID])
        test  = test.drop(columns=[ID])
        if ID in orig.columns:
            orig = orig.drop(columns=[ID])

In [ ]:
# train[TARGET] = train[TARGET].map({True: 1, False: 0})
# orig[TARGET] = orig[TARGET].map({True: 1, False: 0})

obj = train.select_dtypes(include=["object", "category"]).columns.tolist()
bol = train.select_dtypes(include=[bool]).columns.tolist()

cat_cols  = [c for c in obj if c not in [TARGET]]
bool_cols = [c for c in bol if c not in [TARGET]]
num_cols  = [c for c in train.columns if c not in cat_cols+bool_cols+[TARGET]]
base_cols = [c for c in train.columns if c not in [TARGET]]

train = train[base_cols+[TARGET]]
orig  = orig[base_cols+[TARGET]]
test  = test[base_cols]

for (name, df) in {"Train": train, "Test": test, 'Orig': orig}.items(): #
    print(f"{name:5s} shape: {df.shape}")
    
print(f"\nDataFrame type: {type(train)}")
print(f"Total Numerical: {len(num_cols)}")
print(f"Total Categorical: {len(cat_cols)}")
print(f"Total Boolean: {len(bool_cols)}")
print(f"Total base features: {len(base_cols)}")

In [ ]:
float_cols = [
    'amount_usd', 'account_balance_usd',
    'hours_since_last_txn', 'distance_from_home_km',
    'velocity_score', 'merchant_risk_score',
]

int_cols = [
    'customer_age', 'card_age_months', 'time_of_day_hour', 'txn_count_last_24h',
    # 'cvv_retry_count', 'day_of_week', 'prior_disputes',
]

train.head()

In [ ]:
train_len = len(train)
test_len  = len(test)

n_unique = train.nunique().sort_values()
n_unique

In [ ]:
def original_TE(
    orig: pd.DataFrame,
    df: pd.DataFrame,
    features: list=None,
    target: str=None,
    aggs: list=None,
    fill_nan: bool=False,
):

    if features is None or len(features) == 0:
        return df.copy()

    if aggs is None or len(aggs) == 0:
        return df.copy()

    X_df = df.copy()

    orig_cols = []
    maps = {}

    valid_features = [col for col in features if col in orig.columns]

    for col in tqdm(valid_features, desc="original_augmenting"):
        for agg_ in aggs:
            agg_key = agg_.lower()
            new_col = f"oTE_{col}_{agg_key}"

            map_key = (col, agg_key)
            if map_key not in maps:
                try:
                    if agg_key == 'mean':
                        map_df = orig.groupby(col)[target].mean().reset_index(name=new_col)
                    elif agg_key == 'median':
                        map_df = orig.groupby(col)[target].median().reset_index(name=new_col)
                    elif agg_key == 'count':
                        map_df = orig.groupby(col).size().reset_index(name=new_col)
                    elif agg_key == 'nunique':
                        map_df = orig.groupby(col)[target].nunique().reset_index(name=new_col)
                    elif agg_key == 'std':
                        map_df = orig.groupby(col)[target].std().reset_index(name=new_col)
                    elif agg_key == 'skew':
                        map_df = orig.groupby(col)[target].skew().reset_index(name=new_col)
                    elif agg_key == 'max':
                        map_df = orig.groupby(col)[target].max().reset_index(name=new_col)
                    elif agg_key == 'min':
                        map_df = orig.groupby(col)[target].min().reset_index(name=new_col)
                    else:
                        continue
                except Exception as e:
                    print(f"Warning: failed to create map for col={col}, agg={agg_}: {e}")
                    continue

                maps[map_key] = map_df

            map_df = maps.get(map_key)
            if map_df is None:
                continue

            # Merge maps into each split
            X_df = X_df.merge(map_df.astype('float32'), on=col, how='left')

            orig_cols.append(new_col)

    global_mean   = orig[target].mean()
    global_median = orig[target].median()

    def fill_conditionally(df):
        for c in orig_cols:
            if '_mean' in c or '_max' in c or '_min' in c:
                df[c] = df[c].fillna(global_mean)
            elif '_median' in c:
                df[c] = df[c].fillna(global_median)
            else:
                df[c] = df[c].fillna(0)
        return df

    if fill_nan:
        X_df = fill_conditionally(X_df)

    return X_df, orig_cols

print("-- Data augment function ready! --")

In [ ]:
# FEATURES = [c for c in train.columns if c not in [ID, TARGET]]

# combined = pd.concat([train, test, orig], ignore_index=True)
# digit_cols = []

# for col in num_cols:
#     scaled = np.rint(combined[col].fillna(0) * 10**4).astype("int64")
#     for k in range(-2, 3):
#         dig_col = f"{col}_DIGIT_{k}"
#         combined[dig_col] = (scaled // 10**(k+4) % 10).astype('int8').astype('category')
#         digit_cols.append(dig_col)

# train = combined.iloc[:train_len]
# test = combined.iloc[train_len:train_len+test_len].drop(columns=[TARGET])
# orig = combined.iloc[-len(orig):]

# # Check nunique, and find which ones are < 2
# unique_counts = train[digit_cols].nunique()
# drop_cols = unique_counts[unique_counts < 2].index.tolist()

# # update digit cols for keep
# digit_cols = [i for i in set(digit_cols) if i not in set(drop_cols)]

# # drop constant features
# train = train.drop(columns=drop_cols)
# test  = test.drop(columns=drop_cols)
# orig  = orig.drop(columns=drop_cols)

# del combined
# print(f"Total digit features: {len(digit_cols)}\n")

# train[digit_cols].nunique()


In [ ]:
# def digit_keys(df1, cols=float_cols):
#     df = df1.copy()
#     """How each number is written: six label-free keys per float column."""
#     keys = {}
#     for c in cols:
#         units = np.rint(df[c].to_numpy() * 100).astype(np.int64)        # 1371.37 -> 137137
#         integer = units // 100
#         s = pd.Series(integer).astype(str)
#         keys[f"{c}__int"] = integer                                      # 1371
#         keys[f"{c}__int10"] = units // 1000                              # 137
#         keys[f"{c}__tenths"] = units // 10                               # 13713
#         keys[f"{c}__cents"] = units % 100                                # 37
#         keys[f"{c}__lead1"] = (s.str.len() * 10 + s.str[0].astype(int)).to_numpy()     # 4|1
#         keys[f"{c}__lead2"] = (s.str.len() * 100 + s.str[:2].astype(int)).to_numpy()   # 4|13

#     new_df = pd.DataFrame(keys, index=df.index)
#     full_df = pd.concat([df, new_df], axis=1)
#     return full_df, [*keys.keys()]

# train, digit_cols2 = digit_keys(train)
# test, _ = digit_keys(test)

# print(train.shape, test.shape)
# train[digit_cols2].nunique()

In [ ]:
## -- NUMERIC TO CATEGORY for TE -- ⚠️

FEATURES = [c for c in train.columns if c not in [ID, TARGET]]
print("Total features before operation:", len(FEATURES))
combined = pd.concat([train, test, orig], ignore_index=True)

## ------------------------------------------------------------
# -- Duplicate Nums as Cats --
nums_to_cats = []
select_nc = num_cols

for c in tqdm(select_nc, desc="nums_to_cats"):
    n_cat = f"{c}_CAT_"
    combined[n_cat], _ = pd.factorize(combined[c], sort=True)
    combined[n_cat] = combined[n_cat].astype('category')
    nums_to_cats.append(n_cat)

FEATURES = list(set(FEATURES + nums_to_cats))
## ------------------------------------------------------------

train = combined.iloc[:train_len] #[FEATURES+[TARGET]]
test  = combined.iloc[train_len:train_len+test_len].drop(columns=[TARGET])
orig  = combined.iloc[train_len+test_len:]

del combined
print("Total features after operation:", len(FEATURES))

In [ ]:
# ## -- Frequency encoding --
# freq_cols = []
# select_freq = [c for c in base_cols if train[c].nunique() > 2]

# print(f"\nCreating frequencies... ", end="")
# for col in select_freq:
#     freq = pd.concat([train[col], test[col], orig[col]]).value_counts(dropna=False, normalize=True)
#     n = f"{col}_freq"
#     print(n+", ", end="")
#     for df in [train, test, orig]:
#         df[n] = df[col].map(freq).fillna(0).astype("float32")
#     freq_cols.append(n)

# num_cols = list(set(num_cols + freq_cols))
# FEATURES = list(set(FEATURES + freq_cols))

# print(f"\n✓ Total frequency features: {len(freq_cols)}")

# train.head()

In [ ]:
# INTER = []
# combo_select = [c for c in base_cols if train[c].nunique() < 10] + ['Flight Distance_//_1000', 'Age_//_10']

# combo_2_1 = list(itertools.combinations(cat_cols, 2))
# for c1, c2 in tqdm(combo_2_1, desc='Pairwise'):
#     n_col = f"Bi_{c1}_{c2}"
#     train[n_col] = (train[c1].astype(str) + '_' + train[c2].astype(str)).factorize()[0].astype('int32')
#     test[n_col]  = (test[c1].astype(str) + '_' + test[c2].astype(str)).factorize()[0].astype('int32')
#     orig[n_col]  = (orig[c1].astype(str) + '_' + orig[c2].astype(str)).factorize()[0].astype('int32')
#     INTER.append(n_col)

# combo_3_1 = list(itertools.combinations(cat_cols, 3))
# for c1, c2, c3 in tqdm(combo_3_1, desc='Triplewise'):
#     n_col = f"Tri_{c1}_{c2}_{c3}"
#     train[n_col] = ((train[c1].astype(str) + '_' + train[c2].astype(str) + '_' + train[c3].astype(str)).factorize()[0]).astype('int32')
#     test[n_col]  = ((test[c1].astype(str) + '_' + test[c2].astype(str) + '_' + test[c3].astype(str)).factorize()[0]).astype('int32')
#     orig[n_col]  = ((orig[c1].astype(str) + '_' + orig[c2].astype(str) + '_' + orig[c3].astype(str)).factorize()[0]).astype('int32')
#     INTER.append(n_col)

# # combo_2_2 = list(itertools.combinations(digit_cols, 2))
# # for c1, c2 in tqdm(combo_2_2, desc='Pairwise'):
# #     n_col = f"Bi_{c1}_{c2}"
# #     train[n_col] = ((train[c1].astype(str) + '_' + train[c2].astype(str)).factorize()[0]).astype('int32')
# #     test[n_col]  = ((test[c1].astype(str) + '_' + test[c2].astype(str)).factorize()[0]).astype('int32')
# #     orig[n_col]  = ((orig[c1].astype(str) + '_' + orig[c2].astype(str)).factorize()[0]).astype('int32')
# #     INTER.append(n_col)

# # combo_3_list = list(itertools.combinations(digit_cols, 3))
# # for c1, c2, c3 in tqdm(combo_3_list, desc='Triplewise'):
# #     n_col = f"Tri_{c1}_{c2}_{c3}"
# #     train[n_col] = ((train[c1].astype(str) + '_' + train[c2].astype(str) + '_' + train[c3].astype(str)).factorize()[0]).astype('int32')
# #     test[n_col]  = ((test[c1].astype(str) + '_' + test[c2].astype(str) + '_' + test[c3].astype(str)).factorize()[0]).astype('int32')
# #     orig[n_col]  = ((orig[c1].astype(str) + '_' + orig[c2].astype(str) + '_' + orig[c3].astype(str)).factorize()[0]).astype('int32')
# #     INTER.append(n_col)

# # combo_2_1_2 = list(itertools.product(['Age'], cat_cols))
# # for c1, c2 in tqdm(combo_2_1_2, desc='One-to-Many'):
# #     n_col = f"Bi2_{c1}_{c2}"
# #     train[n_col] = ((train[c1].astype(str) + '_' + train[c2].astype(str)).factorize()[0]).astype('int32')
# #     test[n_col]  = ((test[c1].astype(str) + '_' + test[c2].astype(str)).factorize()[0]).astype('int32')
# #     orig[n_col]  = ((orig[c1].astype(str) + '_' + orig[c2].astype(str)).factorize()[0]).astype('int32')
# #     INTER.append(n_col)

# print(f"Total Interaction Features: {len(INTER)}")
# gc.collect()

In [ ]:
_ = [c for c in train.columns if c not in [ID, TARGET]]
FEATURES = list(set(_))

print(f"Train shape {train.shape}")
train.head()

In [ ]:
# ==================================================================  
# OPTIMIZED df_to_libffm  
# ==================================================================  
_FEATURE_MAP = {}  # feature-string -> unique int index (persistent across folds)  
_NEXT_FEATURE_ID = [1]  # mutable counter (1-based)  

def _get_feature_id(feature_str):  
    """Register a feature if new, return its stable ID."""  
    fid = _FEATURE_MAP.get(feature_str)  
    if fid is None:  
        fid = _NEXT_FEATURE_ID[0]  
        _FEATURE_MAP[feature_str] = fid  
        _NEXT_FEATURE_ID[0] += 1  
    return fid  

def df_to_libffm(df, cats, nums, target, save_file="df_converted.libffm", mode="w"):  
    """  
    Vectorized libffm writer (no .iloc per row, single pass).  
    Feature IDs are registered consistently across all folds/files.  
    """  
    field_idx = {c: i for i, c in enumerate(cats)}  
    labels = df[target].astype(int).astype(str).tolist()  
    
    # Fetch all needed column lists ONCE (list conversion is O(1) per column, cached)  
    cols = {}  
    for c in cats:  
        if c != target:  
            vals = df[c]  
            cols[c] = vals.tolist() if hasattr(vals, "tolist") else list(vals)  

    with open(save_file, mode) as f:  
        for i in tqdm(range(len(df)), desc='convert_df_to_libffm'):  
            label = labels[i]  
            parts = [label]
            for field in cats:
                if field == target:
                    continue  
                if field in nums:  
                    fid = _get_feature_id(f"{field}___0")  
                    parts.append(f"{field_idx[field]}:{fid}:{cols[field][i]}")  
                else:  
                    fid = _get_feature_id(f"{field}___x{cols[field][i]}")  
                    parts.append(f"{field_idx[field]}:{fid}:1")  
            f.write(" ".join(parts) + "\n")  

print("- dataframe_to_libffm function ready! -")

In [ ]:
%%writefile convert_libffm_format.py

## -- CONVERT DATAFRAME TO .libffm FORMAT --
import os
import shutil
import tempfile
import warnings
import numpy as np

def df_to_libffm_format(X, y, filepath='data.libffm', fields=None):
    """ Write data to xlearn format (libsvm or libffm). Modified from
    https://github.com/scikit-learn/scikit-learn/blob/a24c8b46/sklearn/datasets/svmlight_format.py

    :param X: array-like
              Feature matrix in numpy or sparse format
    :param y: array-like
              Label in numpy or sparse format
    :param filepath: file location for writing data to
    :param fields: An array specifying fields in each columns of X. It should have same length 
        as the number of columns in X. When set to None, convert data to libsvm format else
        libffm format.
    """

    with open(filepath, "wb") as f_handle:
        X_is_sp = int(hasattr(X, "tocsr"))
        y_is_sp = int(hasattr(y, "tocsr"))

        if X.dtype.kind == 'i':
            value_pattern = u"%d:%d"
        else:
            value_pattern = u"%d:%.16g"

        if fields is not None:
            is_ffm_format = True
            value_pattern = u"%d:" + value_pattern
        else:
            is_ffm_format = False

        if y.dtype.kind == 'i':
            label_pattern = u"%d"
        else:
            label_pattern = u"%.16g"

        line_pattern = u"%s %s\n"

        for i in range(X.shape[0]):
            if X_is_sp:
                span = slice(X.indptr[i], X.indptr[i + 1])
                x_indices = X.indices[span]
                row = zip(fields[x_indices], x_indices, X.data[span]) if is_ffm_format \
                    else zip(x_indices, X.data[span])
            else:
                nz = X[i] != 0
                row = zip(fields[nz], np.where(nz)[0], X[i, nz]) if is_ffm_format \
                    else zip(np.where(nz)[0], X[i, nz])

            if is_ffm_format:
                s = " ".join(value_pattern % (f, j, x) for f, j, x in row)
            else:
                s = " ".join(value_pattern % (j, x) for j, x in row)

            if y_is_sp:
                labels_str = label_pattern % y.data[i]
            else:
                labels_str = label_pattern % y[i]

            f_handle.write((line_pattern % (labels_str, s)).encode('ascii'))

# ML TRAINING

In [ ]:
os.environ['USER'] = 'bill'

In [ ]:
import tempfile

def Trainer_CV(
    model_name, params, arch_type, train_df, test_df,
    features, target, kfold, use_orig=False,
):
    """  
    Fully optimized k-fold CV trainer for xLearn.  
    Improvements: single model reuse, temp dir, vectorized libffm,  
    cleaned file handling, fold-invariant test caching.  
    """  
    t0 = time()  
    print(f"\n===== Starting CV: {model_name} | {get_system_info()} =====")  

    oof_preds  = np.zeros(len(train_df))  
    test_preds = np.zeros(len(test_df))  
    fold_scores = []  

    # Handle optional original table aggregations securely if flags enable them
    if use_orig and "original_TE" in globals():
        feats = features.copy()
        aggs_ = ["count"] # "mean", "count", "std", "skew"] 
        X_tr, orig_cols = original_TE(
            orig, train_df, target=TARGET, features=base_cols, aggs=aggs_, fill_nan=True
        )
        x_test, _ = original_TE(
            orig, test_df, target=TARGET, features=base_cols, aggs=aggs_, fill_nan=True
        )
        feats.extend(orig_cols)
        X = X_tr[feats]
        y = X_tr[target].astype("int32")
        X_ts = x_test[feats].copy()
    else:
        X = train_df[features]
        y = train_df[target].astype("int32")
        X_ts = test_df[features].copy()

    model_class = {  
        'fm':  xlearn.create_fm,  
        'ffm': xlearn.create_ffm,  
        'linear': xlearn.create_linear,  
    }.get(arch_type, xlearn.create_linear)  

    # Dedicated temp dir for all artifact files (auto-cleaned)  
    with tempfile.TemporaryDirectory() as tmpdir:  
        print(f"Using temp workspace: {tmpdir}")

        for idx, (train_idx, val_idx) in enumerate(kfold.split(X, y), 1):  
            print(f"\n### FOLD {idx}/{kfold.n_splits}", end=" | ")  

            X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
            y_train, y_valid = y.iloc[train_idx], y.iloc[val_idx]
            X_test = X_ts.copy()

            # ## -- OPTION B: TE propagate original data --
            # if use_orig:
            #     X_train, _ = orig_TE_data_propagate(
            #         orig,
            #         X_train,
            #         features=base_cols,
            #         target=TARGET,
            #         aggs=orig_aggs,
            #         fill_nan=True,
            #     )
    
            #     X_valid, _ = orig_TE_data_propagate(
            #         orig,
            #         X_valid,
            #         features=base_cols,
            #         target=TARGET,
            #         aggs=orig_aggs,
            #         fill_nan=True,
            #     )

            # ---- Feature lists ----  
            all_cats = X_train.columns
            # all_cats = list(set(cat_cols + [c for c in X_train.columns if c.endswith("_count")]))
            all_nums = [c for c in X_train.columns if c not in all_cats]

            # # ---- Scaling ----  
            # scaler = StandardScaler()
            # X_train[all_nums] = scaler.fit_transform(X_train[all_nums], y_train)  
            # X_valid[all_nums] = scaler.transform(X_valid[all_nums])
            # X_test[all_nums]  = scaler.transform(X_test[all_nums])

            if idx == 1:  
                print(f"shape: {X_train.shape}")  
                display(X_train.head(3))  

            # ---- Write libffm (vectorized, temp dir) ----  
            p_train = os.path.join(tmpdir, f"X_train_{idx}.libffm")  
            p_val   = os.path.join(tmpdir, f"X_val_{idx}.libffm")  
            p_test  = os.path.join(tmpdir, f"X_test_{idx}.libffm")  
            p_oof   = os.path.join(tmpdir, f"oof_{idx}.txt")  
            p_pred  = os.path.join(tmpdir, f"test_{idx}.txt")  
            p_model = os.path.join(tmpdir, f"model_{idx}.out")  

            df_to_libffm(pd.concat([X_train, y_train], axis=1),  
                         all_cats, all_nums, target, p_train, mode='w')  
            df_to_libffm(pd.concat([X_valid, y_valid], axis=1),  
                         all_cats, all_nums, target, p_val, mode='w')  

            _t = np.random.randint(0, 2, size=len(X_test))
            df_to_libffm(pd.concat([X_test, pd.Series(_t,index=X_test.index,name=target)], axis=1),  
                         all_cats, all_nums, target, p_test, mode='w')   
        
            # TRAINING model with validation
            model = model_class()

            # Options for training -------------------------------
            # model.setQuiet()         # ⚠️ Set xlearn to train model quietly.
            # model.setOnDisk()        # ⚠️ Set xlearn to use on-disk training.
            # model.setNoBin()         # ⚠️ Do not generate bin file.
            # model.setSign()          # ⚠️ Label predictions (0 and 1).
            # model.setSigmoid()       # ⚠️ Probabilities (0 to 1).
            # model.disableLockFree()  # ⚠️ Disable lock-free training.
            # model.disableNorm()      # ⚠️ Disable instance-wise normalization.
            # model.disableEarlyStop() # ⚠️ Disable early-stopping.
            # ----------------------- -----------------------------

            model.setTrain(p_train)
            model.setValidate(p_val) 
            # model.disableLockFree()  # ⚠️ Disable lock-free training.
            # model.disableNorm()      # ⚠️ Disable instance-wise normalization.
            model.fit(params, p_model)

            # SEPARATE prediction model
            model.setTest(p_val)
            model.setSigmoid()
            model.predict(p_model, p_oof)
            oof_preds[val_idx] = pd.read_csv(p_oof, sep=" ", header=None).to_numpy().ravel()  

            # TEST predictions (same model object)  
            model.setTest(p_test)  
            model.predict(p_model, p_pred)  
            test_preds += pd.read_csv(p_pred, sep=" ", header=None).to_numpy().ravel() / kfold.n_splits

            # ---- Fold score ----  
            fold_score = roc_auc_score(y_valid, oof_preds[val_idx])  
            fold_scores.append(fold_score)  
            print(f"{CFG['YELLOW']} FOLD {idx} AUC: {fold_score:.6f} {CFG['RESET']}")  

            # ---- Cleanup handled automatically by TemporaryDirectory ----  
            # (files vanish when context exits; no manual sweeps needed)  

            # ---- Memory hygiene ----  
            # del X_train, X_valid, y_train, y_valid 
            gc.collect()  

    ## - Average test predictions -
    # test_preds /= kfold.n_splits 

    ## - CV summary -  
    print("\n==================================================")  
    print(f"{kfold.n_splits}-FOLD CV: {model_name} | {X_train.shape}")
    print("==================================================")  
    for i, score in enumerate(fold_scores, 1):  
        print(f" • Fold {i} AUC: {score:.6f}")  

    oof_score = np.round(roc_auc_score(y, oof_preds), 6)  
    avg_score = np.round(np.mean(fold_scores), 6)  
    std_score = np.round(np.std(fold_scores), 6) 

    print("-------------------------------------------------|")  
    print(f"Overall AUC: {oof_score}")
    print(f"Average AUC: {avg_score} ± {std_score}")   
    print("-------------------------------------------------|")  
    print(f"{(time()-t0)/60:.2f} mins\n")  

    return {
        'oof_preds': oof_preds,
        'test_preds': test_preds,
        'score': avg_score,
        # 'model': model,
        # 'val_data': X_valid,
    }

print("⚙️ Trainer function ready ⚙️")

In [ ]:
all_predictions = {}

MULTI_SEEDS = [42, 777, 1234, 24611, 0]

FOLDS = 5
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

USE_FULL_TRAIN = True
x_sample, x_sample2 = train_test_split(train, train_size=300_000, stratify=train[TARGET], random_state=0)
train_X = train.copy() if USE_FULL_TRAIN else x_sample.reset_index(drop=True)

print(train_X.shape, type(train_X))
train_X.head()

In [ ]:
# !rm -r /kaggle/working/

In [ ]:
## -- Field-aware Factorization Machines (FFM) --

ffm_params = {
    'task': "binary", # "binary", "reg",
    'opt': "adagrad",
    'metric': "auc",
    'lr': 0.1,
    'epoch': 50,
    'stop_window': 5,
    # 'nthread': 3,
    # 'init': 0.8,  # default 0.66
    # 'alpha': 0.2,
    # 'lambda': 0.0002,
    # 'block_size': block_size,
}

n = f"xLEARNffm({train_X.shape[1]})"

# for VALUE in [0.03, 0.04, 0.05, 0.1]:
#     ffm_params['lr'] = VALUE
#     n = "FFM_" + str(VALUE)
all_predictions[n] = Trainer_CV(
    model_name=n,
    params=ffm_params,
    arch_type="ffm",
    train_df=train_X,
    test_df=test,
    features=FEATURES,
    target=TARGET,
    kfold=skf,
    # use_orig=True,
) 

In [ ]:
# - num_to_cats + original(mean, count) + digits + feat_eng

# ==================================================
# 5-FOLD CV: xLEARNffm(34)_lr0.1 | (559708, 33)
# ==================================================
#  • Fold 1 AUC: 0.958785
#  • Fold 2 AUC: 0.957702
#  • Fold 3 AUC: 0.959640
#  • Fold 4 AUC: 0.958877
#  • Fold 5 AUC: 0.958651
# -------------------------------------------------|
# Overall AUC: 0.958726
# Average AUC: 0.958731 ± 0.000619
# -------------------------------------------------|
# 13.02 mins

# ==================================================
# 5-FOLD CV: xLEARNffm(22)_all | (559708, 21)
# ==================================================
#  • Fold 1 AUC: 0.958741
#  • Fold 2 AUC: 0.957525
#  • Fold 3 AUC: 0.959481
#  • Fold 4 AUC: 0.958875
#  • Fold 5 AUC: 0.958563
# -------------------------------------------------|
# Overall AUC: 0.958626
# Average AUC: 0.958637 ± 0.000636
# -------------------------------------------------|
# 3.25 mins

In [ ]:
# !rm -r /kaggle/working/

In [ ]:
## -- Factorization Machines (FM) --
fm_params = {
    'task': "binary", # "binary", "reg",
    'opt': "adagrad",
    'metric': "auc",
    'lr': 0.5,
    'epoch': 120,
    'stop_window': 5,
    # 'nthread': os.cpu_count(),
    # 'init': 0.8,  # default 0.66
    # 'alpha': 0.02,
    # 'lambda': 0.002,
    # 'block_size': block_size,
}

n = f"xLEARN_fm({train_X.shape[1]})"

# for VALUE in [0.05, 0.1, 0.2]:
#     fm_params['lr'] = VALUE
#     n = "FM_" + str(VALUE)
all_predictions[n] = Trainer_CV(
    model_name=n,
    params=fm_params,
    arch_type="fm",
    train_df=train_X,
    test_df=test,
    features=FEATURES,
    target=TARGET,
    kfold=skf,
    # use_orig=True,
) 

In [ ]:
# ==================================================
# 5-FOLD CV: xLEARN_fm(34)_0.5__ | (559708, 33)
# ==================================================
#  • Fold 1 AUC: 0.955716
#  • Fold 2 AUC: 0.954447
#  • Fold 3 AUC: 0.956095
#  • Fold 4 AUC: 0.955516
#  • Fold 5 AUC: 0.955058
# -------------------------------------------------|
# Overall AUC: 0.95536
# Average AUC: 0.955366 ± 0.000569
# -------------------------------------------------|
# 9.59 mins

In [ ]:
# !rm -r /kaggle/working/

In [ ]:
## -- Linear Terms --
lin_params = {
    'task':   "binary", # "binary", "reg",
    'opt':    "adagrad",
    'metric': "auc",
    'lr':      0.1,
    'epoch':   50,
    'stop_window': 5,
    # 'nthread': os.cpu_count(),
    # 'init': 0.8,  # default 0.66
    # 'alpha': 0.2,
    # 'lambda': 0.0001,
    # 'block_size': block_size,
}

n = f"xLEARNlr({train_X.shape[1]})"

# for VALUE in [0.02, 0.03, 0.04]:
#     lin_params['lr'] = VALUE
#     n = "LR" + str(VALUE)
all_predictions[n] = Trainer_CV(
    model_name=n,
    params=lin_params,
    arch_type="linear",
    train_df=train_X,
    test_df=test,
    features=FEATURES,
    target=TARGET,
    kfold=skf,
    # use_orig=True,
) 

In [ ]:
# ==================================================
# 5-FOLD CV: xLEARNlr(34)_0.1___ | (559708, 33)
# ==================================================
#  • Fold 1 AUC: 0.951650
#  • Fold 2 AUC: 0.950369
#  • Fold 3 AUC: 0.951817
#  • Fold 4 AUC: 0.951116
#  • Fold 5 AUC: 0.950976
# -------------------------------------------------|
# Overall AUC: 0.951176
# Average AUC: 0.951185 ± 0.000516
# -------------------------------------------------|
# 5.22 mins


In [ ]:
# !rm -r /kaggle/working/

In [ ]:
## -- Get Scores --
all_scores = {k: v.get("score") for k, v in all_predictions.items() if "score" in v}
results_df = pd.Series(all_scores).sort_values(ascending=False) #[:10]

print(results_df)

In [ ]:
## -- Results --
max_val = results_df.max()
palette = [CRIM if c == max_val else MUTED for c in results_df.values]

plt.figure(figsize=(16, 5))

ax1 = sns.barplot(
    x=results_df.values,
    y=results_df.index,
    palette=palette,
    hue=results_df.index,
    orient="h",
    legend=False,
)

for i, s1 in enumerate(results_df.values):
    ax1.text(
        s1+1e-4,
        float(i),
        f"{results_df.index[i]}{s1:.6f}" if s1 == max_val else f"{results_df.index[i]}{s1:.6f}",
        ha="left",
        va='center_baseline',
        size=12,
        weight="semibold",
        color=CRIM if s1 == max_val else MUTED,
    )

plt.xlim((0.95, 0.962))
plt.title(f"-- {FOLDS}-Fold model scores --", weight="semibold", size=14)

plt.tight_layout()
plt.show()

In [ ]:
## -- Get oof predictions --
oof_predictions = []

for i, (k, v) in enumerate(all_predictions.items()):
    for key_, data_ in v.items():
        if key_ == "oof_preds":
            n = f"{k}_{str(all_scores[k]).split('.')[1]}"
            np.save(f"oof_{n}.npy", data_)
            print(f"{n} saved!")

            ## -- Plot distributions --
            _, axs = plt.subplots(1, 3, figsize=(16, 5)) 
            CalibrationDisplay.from_predictions(train_X[TARGET], data_, n_bins=10, name=n, ax=axs[0])
            axs[0].set_title("Calibration")

            RocCurveDisplay.from_predictions(train_X[TARGET], data_, name=n, ax=axs[1])
            axs[1].set_title("ROC-Curve")

            sns.histplot(data_, ax=axs[2], bins=50)
            axs[2].set_title("Distribution")
            
            plt.tight_layout()
            plt.show()

            print()

In [ ]:
## -- Save test predictions/submissions --
model_results = {}

for i, (k, v) in enumerate(all_predictions.items()):
    for j, (x, y) in enumerate(v.items()):
        if x == "test_preds":
            ## -- Base submission file --
            n = f"{k}_{str(all_scores[k]).split('.')[1]}"
            np.save(f"test_{n}.npy", y)

            submit[TARGET] = y
            submit.to_csv(f"submit_{n}.csv", index=False)
            print(f"{n:>25s} saved! {y.shape}")

            model_results[n[:-7]] = y

print()
model_results = pd.DataFrame(model_results)
model_results

In [ ]:
mdls = model_results.shape[1]

if mdls > 1:
    # display(model_results.corr("spearman"))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(model_results.corr("spearman"), annot=True, fmt=".4g", cbar=False, ax=ax1)
    ax1.set_title('Heatmap Correlation (Spearman)', fontsize=14, fontweight='bold')
    
    sns.ecdfplot(data=model_results, linewidth=2, ax=ax2)
    ax2.set_title(f"CDF Comparison of ({mdls}) Probability Distributions", fontsize=14, fontweight='bold')
    ax2.set_xlabel('Probability Value', fontsize=12)
    ax2.set_ylabel('Cumulative Proportion (CDF)', fontsize=12)
    ax2.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout(pad=2.0)
    plt.show()

In [ ]:
# !rm -r /kaggle/working/